# Train a story model from scratch

This notebook builds a decoder-only Transformer, trains it on next-token prediction and fine-tunes it to write stories with constraints. It assumes basic Python, tensor shapes and gradient descent.

- The default `cpu` profile builds a small, deterministic fixture without a network connection. It exercises every core stage and checks that the implementation learns.
- The `story` profile uses one CUDA GPU or Apple GPU through MPS. It streams pinned revisions of TinyStories for pretraining and TinyStories-Instruct for post-training. A smaller microbatch and more accumulation steps let the same model run with less device memory.
- The notebook implements the tokenizer, Transformer, optimizer and decoding logic. The CPU fixture uses the explicit attention and loss formulas in the notebook. The GPU run uses PyTorch kernels for the same masked attention and cross entropy.

In a Python 3.12 environment, `python -m pip install torch jupyterlab datasets==3.6.0 "huggingface_hub<1"` installs the notebook dependencies. For NVIDIA, the [PyTorch installer](https://pytorch.org/get-started/locally/) provides the command for the CUDA build. When Jupyter starts with `COOK_ML_PROFILE=story jupyter lab`, the kernel uses the GPU profile. `COOK_ML_DEVICE` can select `cuda` or `mps` explicitly. The notebook saves checkpoints under `runs/llm-fundamentals/`. `COOK_ML_RESUME=1` resumes a run with matching training settings and data. `COOK_ML_LABS=1` runs the extra experiments.

Pretraining teaches the model to continue a story. Supervised fine-tuning teaches a copy to write a story that follows a request. A request can combine words to use, a sentence to include, a plot summary and features such as dialogue or a twist. Evaluation gives the same held-out requests to both models. Python checks the words and sentences, and the notebook prints responses for reviewing plot and story features.

Hardware for the `story` profile

- **Apple Silicon Mac.** macOS 15 or later for MPS with [PyTorch 2.15](https://pytorch.org/blog/pytorch-2-15-release-blog/#deprecations-and-backwards-incompatible-changes). An estimated starting point is 8 GB of unified memory. The training times for the larger run are reported on the project page.
- **NVIDIA GPU.** Examples include the desktop GeForce RTX 3060 (12 GB) and GeForce RTX 4060 (8 GB). 4 GB of GPU memory is an estimate based on the Mac run. The notebook uses bf16 autocast when PyTorch reports support and float32 otherwise.
- **Both.** Python 3.12, a network connection to stream the datasets, and about 2 GB of free disk on a Mac (more on Linux or Windows, where the CUDA build of PyTorch is several gigabytes).

The `cpu` profile runs a small fixture without a GPU or a network connection.

The public data comes from [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) and [TinyStories-Instruct](https://huggingface.co/datasets/roneneldan/TinyStoriesInstruct). The notebook fetches both datasets by revision. The project's CPU fixture is generated directly in the notebook.

## Choose a run profile

The same implementation runs on a tiny offline corpus and on a streamed TinyStories subset.

In [ ]:
import copy
import hashlib
import heapq
import math
import os
import random
import re
import time
import unicodedata
from array import array
from collections import Counter, defaultdict
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

@dataclass
class Config:
    vocab_size: int
    width: int
    layers: int
    heads: int
    kv_heads: int
    ff_width: int
    context: int
    micro_batch: int
    accumulation: int
    pretrain_steps: int
    sft_steps: int
    lr: float
    token_budget: int
    sft_train_size: int

    def __post_init__(self):
        if self.width % self.heads or self.heads % self.kv_heads:
            raise ValueError("Width must divide into heads, and query heads into KV groups.")
        if (self.width // self.heads) % 2:
            raise ValueError("RoPE needs an even head width.")
        if self.vocab_size < 260 or self.context < 2:
            raise ValueError("Reserve space for all bytes, four special tokens, and a target.")

CPU = Config(
    vocab_size=512, width=64, layers=2, heads=4, kv_heads=2, ff_width=192,
    context=80, micro_batch=8, accumulation=1, pretrain_steps=160, sft_steps=1000,
    lr=0.003, token_budget=0, sft_train_size=0,
)
STORY = Config(
    vocab_size=4096, width=384, layers=6, heads=6, kv_heads=2, ff_width=1024,
    context=512, micro_batch=4, accumulation=4, pretrain_steps=8000, sft_steps=3500,
    lr=0.0006, token_budget=50_000_000, sft_train_size=60_000,
)
PROFILES = {"cpu": CPU, "story": STORY}
PROFILE = os.environ.get("COOK_ML_PROFILE", "cpu")
cfg = PROFILES[PROFILE]
device_name = os.environ.get("COOK_ML_DEVICE", "auto")
if device_name == "auto":
    if PROFILE == "cpu":
        device_name = "cpu"
    elif torch.cuda.is_available():
        device_name = "cuda"
    elif torch.backends.mps.is_available():
        device_name = "mps"
    else:
        raise RuntimeError("The story profile needs CUDA or Apple MPS. The cpu profile runs offline.")
DEVICE = torch.device(device_name)
USE_BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
SEED = 42
torch.manual_seed(SEED)
if DEVICE.type == "cpu":
    # The small fixture is quick on one thread and gives repeatable CPU results.
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
RUN_DIR = Path(os.environ.get("COOK_ML_OUTPUT", "runs/llm-fundamentals")) / PROFILE
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_LABS = os.environ.get("COOK_ML_LABS", "0") == "1"
print(f"{PROFILE} | {DEVICE} | {'bf16 autocast' if USE_BF16 else 'fp32'}")

## Load pinned TinyStories or the offline corpus

[TinyStories](https://arxiv.org/abs/2305.07759) holds about 2.1 million short stories that GPT-3.5 and GPT-4 wrote with words a three- or four-year-old knows. The vocabulary and grammar are small enough that an 11-million-parameter model can learn to write stories that hang together, which ordinary web text would not allow at this size. The GPU run streams the original train and validation splits at a pinned revision, so every run sees the same text. The CPU run builds short template stories instead, such as "Sam saw a duck by the pond. They found a kite. The duck took the kite home."

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#the-pretraining-corpus)

In [ ]:
DATASET = "roneneldan/TinyStories"
REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"

def normalize_story(text):
    return " ".join(unicodedata.normalize("NFC", text).split())

def story_id(text):
    return hashlib.sha256(normalize_story(text).casefold().encode("utf-8")).hexdigest()

def fixture_stories(count):
    names = ("Mia", "Leo", "Ivy", "Sam", "Bo", "June", "Nora", "Max")
    animals = ("fox", "owl", "cat", "dog", "duck", "frog", "mouse", "bear")
    objects = ("lamp", "kite", "bell", "shell", "coat", "book", "key", "hat")
    places = ("bridge", "pond", "hill", "garden", "farm", "lake", "park", "trail")
    for name, animal, obj, place in random.Random(SEED).sample(
        [(n, a, o, p) for n in names for a in animals for o in objects for p in places], count
    ):
        yield f"{name} saw a {animal} by the {place}. They found a {obj}. The {animal} took the {obj} home."

def public_stories(split):
    from datasets import load_dataset

    rows = load_dataset(DATASET, revision=REVISION, split=split, streaming=True)
    for row in rows:
        yield row["text"]

def deduplicate(stories, seen):
    for raw_story in stories:
        text = normalize_story(raw_story)
        if not text:
            continue
        key = story_id(text)
        if key not in seen:
            seen.add(key)
            yield key, text

## Reserve stories before pretraining

The notebook sets aside 1,500 stories from TinyStories validation to measure the base model. It hashes normalized story text and leaves buckets 0 through 17 out of pretraining. TinyStories-Instruct reuses stories from TinyStories, so the same hash identifies eligible fine-tuning stories even when their formatting differs.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#deduplication-and-quality-filtering)

In [ ]:
from itertools import islice

def story_bucket(key):
    return int(key[:8], 16) % 100

seen_stories = set()
if PROFILE == "cpu":
    fixture = list(fixture_stories(3000))
    base_validation = [text for _, text in deduplicate(fixture[:120], seen_stories)]
    candidates = iter(fixture[120:])
else:
    base_validation = [
        text for _, text in islice(deduplicate(public_stories("validation"), seen_stories), 1500)
    ]
    candidates = public_stories("train")

# Leave about 18% of story IDs available for instruction data.
eligible = (text for key, text in deduplicate(candidates, seen_stories) if story_bucket(key) >= 18)
tokenizer_story_count = 1800 if PROFILE == "cpu" else 20_000
pretrain_stories = list(islice(eligible, tokenizer_story_count))
assert len(pretrain_stories) == tokenizer_story_count and base_validation
print({"tokenizer stories": len(pretrain_stories), "base validation": len(base_validation)})

def pretraining_stories():
    yield from pretrain_stories
    if PROFILE == "story":
        for story in eligible:
            pretrain_stories.append(story)
            yield story

## Start the vocabulary with every byte

UTF-8 gives the tokenizer a reversible representation of arbitrary text, including characters outside the training corpus.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/tokenization-and-embeddings/#byte-level-fallback)

In [ ]:
BOS, EOS, USER, ASSISTANT = range(256, 260)
FIRST_MERGE = 260
PIECES = re.compile(r"\s+|\w+|[^\w\s]+", re.UNICODE)

def byte_pieces(text):
    return [tuple(piece.encode("utf-8")) for piece in PIECES.findall(text)]

def merge_pair(tokens, pair, merged):
    output = []
    index = 0
    while index < len(tokens):
        if tuple(tokens[index:index + 2]) == pair:
            output.append(merged)
            index += 2
        else:
            output.append(tokens[index])
            index += 1
    return tuple(output)

## Train byte pair merges

A training step picks the most common adjacent pair. Word frequencies let repeated pieces share the same work.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/tokenization-and-embeddings/#byte-pair-encoding)

In [ ]:
def train_bpe(texts, vocab_size):
    words = Counter(piece for text in texts for piece in byte_pieces(text))
    sequences, weights = zip(*words.items())
    sequences = list(sequences)
    counts, locations = Counter(), defaultdict(set)

    def update(word_index, tokens, direction):
        pairs = Counter(zip(tokens, tokens[1:]))
        for pair, repeats in pairs.items():
            counts[pair] += direction * repeats * weights[word_index]
            if direction > 0:
                locations[pair].add(word_index)
            else:
                locations[pair].discard(word_index)
        return pairs.keys()

    for index, tokens in enumerate(sequences):
        update(index, tokens, 1)
    heap = [(-count, pair) for pair, count in counts.items()]
    heapq.heapify(heap)
    merges = []
    while heap and FIRST_MERGE + len(merges) < vocab_size:
        negative_count, pair = heapq.heappop(heap)
        if -negative_count != counts[pair] or not counts[pair]:
            continue
        merged = FIRST_MERGE + len(merges)
        merges.append(pair)
        changed = set()
        for index in tuple(locations[pair]):
            previous = sequences[index]
            current = merge_pair(previous, pair, merged)
            changed.update(update(index, previous, -1))
            changed.update(update(index, current, 1))
            sequences[index] = current
        for other in changed:
            if counts[other]:
                heapq.heappush(heap, (-counts[other], other))
    return merges

## Encode and decode with learned merges

Encoding follows the order of the learned merges. Decoding joins token bytes before converting back to text.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/tokenization-and-embeddings/#text-units)

In [ ]:
class ByteBPE:
    def __init__(self, merges):
        self.merges = [tuple(pair) for pair in merges]
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.tokens = [bytes([i]) for i in range(256)] + [b""] * 4
        for left, right in self.merges:
            self.tokens.append(self.tokens[left] + self.tokens[right])
        self.cache = {}

    def merge_piece(self, piece):
        if piece not in self.cache:
            tokens = piece
            while len(tokens) > 1:
                pair = min(zip(tokens, tokens[1:]), key=lambda p: self.ranks.get(p, math.inf))
                if pair not in self.ranks:
                    break
                tokens = merge_pair(tokens, pair, FIRST_MERGE + self.ranks[pair])
            if len(self.cache) < 100_000:
                self.cache[piece] = tokens
            return tokens
        return self.cache[piece]

    def encode(self, text):
        return [token for piece in byte_pieces(text) for token in self.merge_piece(piece)]

    def decode(self, token_ids):
        return b"".join(self.tokens[token] for token in token_ids).decode("utf-8", errors="replace")

    def document(self, text):
        return [BOS, *self.encode(text), EOS]

tokenizer = ByteBPE(train_bpe(pretrain_stories[:20_000], cfg.vocab_size))
sample = "A fox 🦊 crosses a bridge."
assert tokenizer.decode(tokenizer.encode(sample)) == sample
print(f"{len(tokenizer.tokens):,} assigned token IDs | {sample} → {tokenizer.encode(sample)}")

## Pack documents without sharing context

Each token receives a document ID. The final token of one document never learns to predict the first token of the next.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#sequence-packing)

In [ ]:
IGNORE = -100

def pack_documents(documents, context, token_budget=None):
    packed_x, packed_y, packed_segment = array("i"), array("i"), array("i")
    x, y, segments = [], [], []
    segment = 0
    used_targets = 0

    def flush():
        padding = context - len(x)
        packed_x.extend(x + [EOS] * padding)
        packed_y.extend(y + [IGNORE] * padding)
        packed_segment.extend(segments + [-1] * padding)
        x.clear()
        y.clear()
        segments.clear()

    for inputs, targets in documents:
        for start in range(0, len(inputs), context):
            length = context if token_budget is None else min(context, token_budget - used_targets)
            chunk_x = inputs[start:start + length]
            chunk_y = targets[start:start + length]
            if len(x) + len(chunk_x) > context:
                flush()
                segment = 0
            x.extend(chunk_x)
            y.extend(chunk_y)
            segments.extend([segment] * len(chunk_x))
            segment += 1
            used_targets += sum(target != IGNORE for target in chunk_y)
            if token_budget is not None and used_targets >= token_budget:
                break
        if token_budget is not None and used_targets >= token_budget:
            break
    if x:
        flush()
    assert packed_x and used_targets
    return tuple(
        torch.frombuffer(buffer, dtype=torch.int32).reshape(-1, context)
        for buffer in (packed_x, packed_y, packed_segment)
    ), used_targets

## Shift next-token targets by one

The input at position t pairs with the true next token at position t+1.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/language-modeling-and-next-token-prediction/#teacher-forcing-and-exposure-bias)

In [ ]:
def shift_document(ids):
    return ids[:-1], ids[1:]

train_batches, train_tokens = pack_documents(
    (shift_document(tokenizer.document(story)) for story in pretraining_stories()),
    cfg.context, token_budget=cfg.token_budget or None,
)
if cfg.token_budget and train_tokens < cfg.token_budget:
    raise ValueError(f"Training stream ended after {train_tokens:,} of {cfg.token_budget:,} targets.")
print(f"{len(pretrain_stories):,} stories | {train_batches[0].shape[0]:,} packed rows | "
      f"{train_tokens:,} real training targets in the pool")

## Compute stable next-token cross entropy

Subtract the log normalizer from the target logit. Padding and masked prompt positions contribute zero.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/language-modeling-and-next-token-prediction/#cross-entropy-loss)

In [ ]:
def cross_entropy_loss(logits, targets):
    logits = logits.float()
    mask = targets != IGNORE
    safe_targets = targets.masked_fill(~mask, 0)
    chosen = logits.gather(-1, safe_targets[..., None]).squeeze(-1)
    losses = torch.logsumexp(logits, dim=-1) - chosen
    return losses.masked_fill(~mask, 0.0), mask

toy_logits = torch.tensor([[[1.0, 3.0, 2.0], [1.0, 3.0, 2.0]]])
toy_targets = torch.tensor([[1, IGNORE]])
toy_loss, toy_mask = cross_entropy_loss(toy_logits, toy_targets)
expected = math.log(1 + math.exp(-1) + math.exp(-2))
assert abs(toy_loss.sum().item() - expected) < 1e-6
print(round(toy_loss.sum().item(), 4))

## Fit a smoothed bigram baseline

The baseline counts adjacent tokens and adds a small count to unseen pairs.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/language-modeling-and-next-token-prediction/#n-gram-models)

In [ ]:
def fit_bigram(documents):
    pairs, origins = Counter(), Counter()
    for ids in documents:
        pairs.update(zip(ids, ids[1:]))
        origins.update(ids[:-1])
    return pairs, origins

def bigram_nll(ids, counts, vocab_size, alpha=0.1):
    pairs, origins = counts
    return -sum(math.log((pairs[left, right] + alpha) / (origins[left] + alpha * vocab_size))
                for left, right in zip(ids, ids[1:]))

bigram = fit_bigram(tokenizer.document(story) for story in pretrain_stories[:10_000])

## Build the causal mask

A token can read its own position and earlier positions from the same document. This three-token example is also a runnable check.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#the-causal-mask)

In [ ]:
def causal_mask(
    query_positions, key_positions, query_segments=None, key_segments=None
):
    visible = key_positions[..., None, :] <= query_positions[..., :, None]
    if query_segments is not None:
        same_document = query_segments[..., :, None] == key_segments[..., None, :]
        visible = visible & same_document
    return visible

positions = torch.arange(3)
visible = causal_mask(positions, positions)
expected = [[True, False, False], [True, True, False], [True, True, True]]
assert visible.tolist() == expected
segments = torch.tensor([0, 0, 1])
assert not causal_mask(positions, positions, segments, segments)[2, 1]
print(visible.int())

## Rotate query and key pairs

Rotary position embedding rotates two adjacent channels at a time. Cache entries keep their original position.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#rotary-position-embedding)

In [ ]:
def apply_rope(x, positions, base=10_000.0):
    head_width = x.shape[-1]
    exponent = torch.arange(0, head_width, 2, device=x.device, dtype=torch.float32) / head_width
    theta = positions.float()[..., None] * base ** -exponent
    cos, sin = theta.cos()[:, None], theta.sin()[:, None]
    even, odd = x.float()[..., 0::2], x.float()[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2).to(x.dtype)

rotary_probe = torch.randn(1, 2, 3, 4)
assert torch.allclose(rotary_probe.norm(dim=-1), apply_rope(rotary_probe, positions[None]).norm(dim=-1))

## Turn logits into probabilities

Moving the largest logit to zero keeps exponentials finite without changing the result.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/language-modeling-and-next-token-prediction/#logits-and-softmax)

In [ ]:
def stable_softmax(logits, dim=-1):
    logits = logits.float()
    shifted = logits - logits.amax(dim=dim, keepdim=True)
    weights = shifted.exp()
    return weights / weights.sum(dim=dim, keepdim=True)

assert torch.allclose(stable_softmax(torch.tensor([1000., 1001.])), torch.tensor([0.2689414, 0.7310586]))

## Score keys and mix values

Scale query-key dot products by the square root of the head width, then mix the values using masked probabilities.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#queries-keys-and-values)

In [ ]:
def attend(q, k, v, visible):
    if q.device.type != "cpu":
        return F.scaled_dot_product_attention(q, k, v, attn_mask=visible[:, None])
    scores = (q @ k.transpose(-1, -2)).float() / math.sqrt(q.shape[-1])
    probabilities = stable_softmax(scores.masked_fill(~visible[:, None], -torch.inf))
    return probabilities.to(v.dtype) @ v

## Write a linear projection with a weight matrix

A projection multiplies the final dimension by a learned matrix. The same operation is used for each attention and feed-forward matrix.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#multi-head-attention)

In [ ]:
class Linear(nn.Module):
    def __init__(self, input_width, output_width):
        super().__init__()
        self.weight = nn.Parameter(0.02 * torch.randn(output_width, input_width))

    def forward(self, x):
        return x @ self.weight.T

## Project grouped queries, keys and values

The model keeps two KV heads per layer. Each KV head serves a group of query heads while the cache stores it once.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#head-sharing)

In [ ]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads, self.kv_heads = config.heads, config.kv_heads
        self.head_width = config.width // config.heads
        kv_width = self.kv_heads * self.head_width
        self.q = Linear(config.width, config.width)
        self.k = Linear(config.width, kv_width)
        self.v = Linear(config.width, kv_width)
        self.out = Linear(config.width, config.width)

    def split_heads(self, x, heads):
        return x.unflatten(-1, (heads, self.head_width)).transpose(1, 2)

    def forward(self, x, positions, visible, cache=None, use_cache=False):
        q = apply_rope(self.split_heads(self.q(x), self.heads), positions)
        k = apply_rope(self.split_heads(self.k(x), self.kv_heads), positions)
        v = self.split_heads(self.v(x), self.kv_heads)
        if cache is not None:
            k, v = (torch.cat((previous, current), dim=-2) for previous, current in zip(cache, (k, v)))
        groups = self.heads // self.kv_heads
        mixed = attend(q, k.repeat_interleave(groups, dim=1), v.repeat_interleave(groups, dim=1), visible)
        return self.out(mixed.transpose(1, 2).flatten(-2)), (k, v) if use_cache else None

## Normalize by root mean square

Compute the statistic in float32 and let a learned gain set the size of each channel.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#layer-normalization-and-rmsnorm)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, width, eps=1e-5):
        super().__init__()
        self.gain = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        scaled = xf * torch.rsqrt((xf * xf).mean(dim=-1, keepdim=True) + self.eps)
        return (scaled * self.gain).to(x.dtype)

## Build the gated feed-forward layer

One projection produces content and the other controls how much of that content gets through.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#swiglu)

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, width, ff_width):
        super().__init__()
        self.gate = Linear(width, ff_width)
        self.up = Linear(width, ff_width)
        self.down = Linear(ff_width, width)

    def forward(self, x):
        gate = self.gate(x)
        return self.down((gate * gate.sigmoid()) * self.up(x))

## Compose the pre-norm decoder block

Normalize the input of each sublayer and add its output back to the residual stream.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#the-decoder-block)

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.width)
        self.attention = Attention(config)
        self.ff_norm = RMSNorm(config.width)
        self.feed_forward = SwiGLU(config.width, config.ff_width)

    def forward(self, x, positions, visible, cache=None, use_cache=False):
        delta, new_cache = self.attention(self.attention_norm(x), positions, visible, cache, use_cache)
        x = x + delta
        return x + self.feed_forward(self.ff_norm(x)), new_cache

## Reuse a single embedding table

An input token selects one row. The output head reads the same weights in the opposite direction.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/tokenization-and-embeddings/#weight-tying)

In [ ]:
class TiedEmbedding(nn.Module):
    def __init__(self, vocab_size, width):
        super().__init__()
        self.weight = nn.Parameter(0.02 * torch.randn(vocab_size, width))

    def forward(self, token_ids):
        return self.weight[token_ids]

    def logits(self, x):
        return x @ self.weight.T

## Assemble the story model

The stack turns token IDs into next-token logits. Training supplies document IDs and generation can supply a per-layer cache.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#the-output-head)

In [ ]:
class StoryModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = TiedEmbedding(config.vocab_size, config.width)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.layers)])
        self.final_norm = RMSNorm(config.width)

    def forward(self, tokens, segments=None, cache=None, use_cache=False):
        if cache is not None and (not use_cache or segments is not None):
            raise ValueError("Pass a cache only for continuous generation with use_cache=True.")
        offset = cache[0][0].shape[-2] if cache else 0
        total_length = offset + tokens.shape[-1]
        if total_length > self.config.context:
            raise ValueError("Input exceeds the configured context window.")
        key_positions = torch.arange(total_length, device=tokens.device)[None]
        query_positions = key_positions[:, offset:]
        visible = causal_mask(query_positions, key_positions, segments, segments)
        x = self.embedding(tokens)
        new_cache = []
        for i, block in enumerate(self.blocks):
            x, layer_cache = block(x, query_positions, visible, cache[i] if cache else None, use_cache)
            if use_cache:
                new_cache.append(layer_cache)
        return self.embedding.logits(self.final_norm(x)), new_cache if use_cache else None

## Count parameters and cache bytes

The parameter formula accounts for tied weights and narrower KV projections. Cache size follows from the two KV heads stored per layer.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#parameter-counting)

In [ ]:
def model_accounting(config, dtype_bytes=2):
    d = config.width
    head_width = d // config.heads
    kv_width = config.kv_heads * head_width
    attention = 2 * d * d + 2 * d * kv_width
    feed_forward = 3 * d * config.ff_width
    shared_table = config.vocab_size * d
    params = shared_table + config.layers * (attention + feed_forward + 2 * d) + d
    weight_bytes = params * dtype_bytes
    cache_bytes = 2 * config.layers * config.context * kv_width * dtype_bytes
    # Approximation for dense projections and the two full attention matrix products.
    forward_flops = 2 * (shared_table + config.layers * (attention + feed_forward))
    forward_flops += 4 * config.layers * config.context * d
    return dict(parameters=params, weight_bytes=weight_bytes, cache_bytes=cache_bytes,
                forward_flops_per_token=forward_flops)

model = StoryModel(cfg).to(DEVICE)
accounting = model_accounting(cfg)
assert sum(p.numel() for p in model.parameters()) == accounting["parameters"]
print(accounting)

## Update weights with AdamW

Track the first and second moments explicitly, correct for their initial bias, and apply weight decay separately from the gradient.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#adamw)

In [ ]:
class AdamW:
    def __init__(self, named_parameters, beta1=0.9, beta2=0.95, eps=1e-8, decay=0.1):
        self.parameters = dict(named_parameters)
        self.beta1, self.beta2 = beta1, beta2
        self.eps, self.decay = eps, decay
        self.step_number = 0
        self.first = {name: torch.zeros_like(p) for name, p in self.parameters.items()}
        self.second = {name: torch.zeros_like(p) for name, p in self.parameters.items()}

    @torch.no_grad()
    def step(self, lr):
        self.step_number += 1
        correction1 = 1 - self.beta1 ** self.step_number
        correction2 = 1 - self.beta2 ** self.step_number
        for name, parameter in self.parameters.items():
            grad = parameter.grad
            if grad is None:
                continue
            first, second = self.first[name], self.second[name]
            first.mul_(self.beta1).add_(grad, alpha=1 - self.beta1)
            second.mul_(self.beta2).addcmul_(grad, grad, value=1 - self.beta2)
            if parameter.ndim >= 2:
                parameter.mul_(1 - lr * self.decay)
            step = (first / correction1) / ((second / correction2).sqrt() + self.eps)
            parameter.add_(step, alpha=-lr)

    def zero_grad(self):
        for parameter in self.parameters.values():
            parameter.grad = None

## Warm up and decay the learning rate

The learning rate rises linearly at the start, then falls along a cosine curve.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#learning-rate-schedules)

In [ ]:
def learning_rate(step, steps, peak):
    warmup = max(1, int(steps * 0.08))
    if step < warmup:
        return peak * (step + 1) / warmup
    progress = (step - warmup) / max(1, steps - warmup - 1)
    return peak * (0.1 + 0.9 * (1 + math.cos(math.pi * progress)) / 2)

assert learning_rate(7, 100, 0.001) == 0.001
assert math.isclose(learning_rate(99, 100, 0.001), 0.0001)

## Clip the global gradient norm

If the total gradient length exceeds the threshold, every gradient is scaled by the same factor.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#loss-spikes)

In [ ]:
@torch.no_grad()
def clip_gradients(model, maximum=1.0):
    gradients = [p.grad for p in model.parameters() if p.grad is not None]
    norm = torch.stack([g.float().square().sum() for g in gradients]).sum().sqrt()
    if not norm.isfinite():
        raise FloatingPointError("Gradient norm became non-finite.")
    scale = (maximum / (norm + 1e-6)).clamp(max=1.0)
    for grad in gradients:
        grad.mul_(scale)
    return norm.item()

## Choose precision for matrix products

The GPU profile can use bfloat16 for matrix products. Parameters and optimizer moments stay in float32, as do the loss and softmax calculations.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#mixed-precision)

In [ ]:
def training_precision():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if USE_BF16 else nullcontext()

## Save and restore training state

The checkpoint carries weights, AdamW moments, the sampler position, and the tokenizer. It is written through a temporary file before replacing the previous checkpoint.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#optimizer-memory)

In [ ]:
def save_checkpoint(path, model, optimizer, sampler, step, signature):
    state = {
        "signature": signature, "step": step, "model": model.state_dict(),
        "first": optimizer.first, "second": optimizer.second,
        "step_number": optimizer.step_number, "sampler": sampler.get_state(),
    }
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)

def load_checkpoint(path, model, optimizer, sampler, signature):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if state["signature"] != signature:
        raise ValueError(f"Checkpoint settings differ: {path}")
    model.load_state_dict(state["model"])
    for name in optimizer.parameters:
        optimizer.first[name].copy_(state["first"][name])
        optimizer.second[name].copy_(state["second"][name])
    optimizer.step_number = state["step_number"]
    sampler.set_state(state["sampler"])
    return state["step"]

## Train with accumulated gradients

One update averages the loss over `accumulation × micro_batch` packed rows. The story profile uses 4 × 4 rows of 512 slots, so an update can include up to 8,192 targets. It shuffles the packed rows before each pass through a training pool. Pretraining uses 8,000 updates and fine-tuning uses 3,500. Accumulation lets a GPU with little memory hold only 4 rows at a time and still take the same update as one batch of 16. The loss is divided by the number of real targets across all four microbatches, so padding and short rows do not change the weighting. A checkpoint is written every 250 updates.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#the-compute-budget)

In [ ]:
def train(model, batches, steps, lr, phase, resume=False):
    optimizer = AdamW((name, p) for name, p in model.named_parameters() if p.requires_grad)
    sampler = torch.Generator().manual_seed(SEED)
    dataset_fingerprint = hashlib.sha256()
    for tensor in batches:
        dataset_fingerprint.update(memoryview(tensor.numpy()).cast("B"))
    signature = {
        "config": asdict(model.config), "phase": phase, "seed": SEED,
        "data": dataset_fingerprint.hexdigest(), "merges": tokenizer.merges,
        "steps": steps, "learning_rate": lr, "bf16": USE_BF16,
    }
    shuffle = PROFILE == "story" and phase in {"base", "sft-instruct"}
    if shuffle:
        signature["sampler"] = "shuffled-epochs"
    path = RUN_DIR / f"{phase}.pt"
    start = load_checkpoint(path, model, optimizer, sampler, signature) if resume and path.exists() else 0
    model.train()
    if start >= steps:
        print(f"{phase}: loaded completed checkpoint {path}")
        return dict(loss=None, tokens=0, seconds=0.0)
    total_loss, total_tokens = 0.0, 0
    started = time.perf_counter()
    rows_per_step = cfg.accumulation * cfg.micro_batch
    steps_per_epoch = len(batches[0]) // rows_per_step
    if shuffle and not steps_per_epoch:
        raise ValueError("The training pool must fill at least one update.")
    previous_epoch = None
    for step in range(start, steps):
        if shuffle:
            epoch, offset = divmod(step, steps_per_epoch)
            if epoch != previous_epoch:
                sampler.manual_seed(SEED + epoch)
                order = torch.randperm(len(batches[0]), generator=sampler)
                previous_epoch = epoch
            first = offset * rows_per_step
            trial = order[first:first + rows_per_step].view(cfg.accumulation, cfg.micro_batch)
        else:
            trial = torch.randint(len(batches[0]), (cfg.accumulation, cfg.micro_batch), generator=sampler)
        total_valid = (batches[1][trial] != IGNORE).sum().item()
        optimizer.zero_grad()
        update_loss = 0.0
        for rows in trial:
            tokens, targets, segments = [t[rows].to(DEVICE, dtype=torch.long) for t in batches]
            with training_precision():
                logits, _ = model(tokens, segments)
                if DEVICE.type == "cpu":
                    losses, _ = cross_entropy_loss(logits, targets)
                    loss = losses.sum() / total_valid
                else:
                    loss = F.cross_entropy(logits.flatten(0, 1).float(), targets.flatten(),
                                           ignore_index=IGNORE, reduction="sum") / total_valid
            loss.backward()
            update_loss += loss.item()
        clip_gradients(model)
        optimizer.step(learning_rate(step, steps, lr))
        total_loss += update_loss * total_valid
        total_tokens += total_valid
        if step == start or (step + 1) % max(1, min(500, steps // 4)) == 0 or step + 1 == steps:
            print(f"{phase} {step + 1:4}/{steps}: loss {update_loss:.3f}")
        if (step + 1) % 250 == 0 or step + 1 == steps:
            save_checkpoint(path, model, optimizer, sampler, step + 1, signature)
    elapsed = time.perf_counter() - started
    print(f"{phase}: {total_tokens:,} valid targets in {elapsed:.1f}s. Checkpoint: {path}")
    return dict(loss=total_loss / max(1, total_tokens), tokens=total_tokens, seconds=elapsed)

## Measure held-out perplexity and bits per byte

Sum loss over all validation targets before dividing. Bits per byte uses raw UTF-8 text bytes and excludes the end marker.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/language-modeling-and-next-token-prediction/#perplexity)

In [ ]:
@torch.inference_mode()
def score_next_tokens(model, batches):
    model.eval()
    nll, text_nll, count = 0.0, 0.0, 0
    for start in range(0, len(batches[0]), cfg.micro_batch):
        x, y, segment = [t[start:start + cfg.micro_batch].to(DEVICE, dtype=torch.long) for t in batches]
        with training_precision():
            logits, _ = model(x, segment)
            losses, valid = cross_entropy_loss(logits, y)
        nll += losses.sum().item()
        text_nll += losses.masked_fill(y == EOS, 0).sum().item()
        count += valid.sum().item()
    return nll, text_nll, count

def evaluate_language_model(model, stories):
    batches, _ = pack_documents(
        (shift_document(tokenizer.document(story)) for story in stories), cfg.context
    )
    nll, text_nll, count = score_next_tokens(model, batches)
    byte_count = sum(len(story.encode("utf-8")) for story in stories)
    return dict(loss=nll / count, perplexity=math.exp(nll / count),
                bits_per_byte=text_nll / (math.log(2) * byte_count), targets=count)

def evaluate_bigram(stories):
    documents = [tokenizer.document(story) for story in stories]
    nll = sum(bigram_nll(ids, bigram, cfg.vocab_size) for ids in documents)
    return math.exp(nll / sum(len(ids) - 1 for ids in documents))

## Train and save the base checkpoint

This is the first training task, next-token prediction. The model starts from random weights, so its loss begins near ln 4,096 ≈ 8.3, the cost of guessing uniformly among 4,096 tokens. The held-out stories are scored before and after training, beside a bigram baseline that only counts adjacent token pairs. The gap between the bigram and the trained model is what the Transformer's attention to earlier context adds. The cell also prints the training throughput, and the peak GPU memory on CUDA.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#the-compute-budget)

In [ ]:
resume = os.environ.get("COOK_ML_RESUME", "0") == "1"
before = evaluate_language_model(model, base_validation)
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
base_run = train(model, train_batches, cfg.pretrain_steps, cfg.lr, "base", resume=resume)
base_metrics = evaluate_language_model(model, base_validation)
print({"untrained": before, "bigram_ppl": evaluate_bigram(base_validation), "pretrained": base_metrics})
if DEVICE.type == "cuda":
    print(f"peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
if base_run["tokens"]:
    print(f"{base_run['tokens'] / base_run['seconds']:,.0f} target tokens/s including checkpoint writes")

## See how temperature reshapes the same logits

Lower temperatures concentrate probability on the highest scores. The sampling cell uses the same division before applying truncation.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#temperature-sampling)

In [ ]:
def temperature_scores(logits, temperature):
    if temperature <= 0:
        raise ValueError("Sampling temperature must be positive.")
    return logits / temperature

example = torch.tensor([1., 2., 3.])
cold = stable_softmax(temperature_scores(example, 0.5))
warm = stable_softmax(temperature_scores(example, 1.5))
assert cold[-1] > warm[-1]

## Sample with temperature, top-k and top-p

Truncation removes low-probability choices before sampling. Greedy decoding uses the largest logit.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#truncation-sampling)

In [ ]:
def choose_token(logits, temperature=0.0, top_k=None, top_p=1.0, generator=None):
    scores = logits.float().clone()
    scores[len(tokenizer.tokens):] = -torch.inf
    scores[[BOS, USER, ASSISTANT]] = -torch.inf
    if temperature == 0:
        return scores.argmax().item()
    if temperature < 0 or not 0 < top_p <= 1 or (top_k is not None and top_k < 1):
        raise ValueError("Temperature, top_k or top_p is out of range.")
    sorted_scores, indices = temperature_scores(scores, temperature).sort(descending=True)
    if top_k is not None:
        sorted_scores[top_k:] = -torch.inf
    probs = stable_softmax(sorted_scores)
    # Keep the first token that takes cumulative mass past top_p.
    probs = probs.masked_fill((probs.cumsum(0) - probs) >= top_p, 0)
    cumulative = (probs / probs.sum()).cumsum(0)
    cumulative[-1] = 1.0
    draw = torch.rand((), generator=generator, device=cumulative.device)
    return indices[torch.searchsorted(cumulative, draw, right=True)].item()

## Reuse keys and values during generation

Prefill reads the prompt once. Cached decoding adds one new token at a time and stops at EOS or the context limit.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#the-key-value-cache)

In [ ]:
@torch.inference_mode()
def generate(model, prefix, max_new=128, temperature=0.0, top_k=None, top_p=1.0, seed=SEED, cached=True):
    model.eval()
    if not prefix or len(prefix) >= model.config.context:
        raise ValueError("The prompt must leave room for at least one generated token.")
    tokens = torch.tensor([prefix], dtype=torch.long, device=DEVICE)
    logits, cache = model(tokens, use_cache=cached)
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    output = []
    for index in range(min(max_new, model.config.context - len(prefix))):
        new_id = choose_token(logits[0, -1], temperature, top_k, top_p, generator)
        if new_id == EOS:
            break
        output.append(new_id)
        next_token = torch.tensor([[new_id]], dtype=torch.long, device=DEVICE)
        if index + 1 >= min(max_new, model.config.context - len(prefix)):
            break
        if cached:
            logits, cache = model(next_token, cache=cache, use_cache=True)
        else:
            tokens = torch.cat((tokens, next_token), dim=1)
            logits, _ = model(tokens)
    return tokenizer.decode(output)

@torch.inference_mode()
def check_cache(model, text):
    ids = torch.tensor([[BOS, *tokenizer.encode(text)]], dtype=torch.long, device=DEVICE)
    full, _ = model(ids)
    _, before_last = model(ids[:, :-1], use_cache=True)
    last, cache = model(ids[:, -1:], cache=before_last, use_cache=True)
    torch.testing.assert_close(last[:, 0], full[:, -1], rtol=1e-4, atol=2e-5)
    assert cache[0][0].shape[1] == model.config.kv_heads

check_cache(model, "A fox saw a bridge.")
starter = "One day" if PROFILE == "story" else "Mia saw"
print(starter + generate(model, [BOS, *tokenizer.encode(starter)], max_new=256))

## Read published instruction records

The TinyStories-Instruct files put labels such as Words or Summary above each story. An end-of-text marker separates the records. Reading one record at a time lets the notebook sample the published files without saving a local copy.

In [ ]:
INSTRUCT_DATASET = "roneneldan/TinyStoriesInstruct"
INSTRUCT_REVISION = "ee050ed1f8720795be342921335e821856a2b42e"

FEATURE_NAMES = {
    "Dialogue": "dialogue", "BadEnding": "a bad ending", "MoralValue": "a moral",
    "Twist": "a plot twist", "Foreshadowing": "foreshadowing", "Conflict": "a conflict",
}
PROMPT_LABELS = {
    "words": "Use these words", "sentence": "Include this sentence",
    "summary": "Follow this plot", "features": "Include these features",
}
WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def public_instruction_records(split):
    from urllib.request import urlopen

    file = f"TinyStories-Instruct-{split}.txt"
    url = f"https://huggingface.co/datasets/{INSTRUCT_DATASET}/resolve/{INSTRUCT_REVISION}/{file}"
    with urlopen(url, timeout=60) as stream:
        lines = []
        for raw_line in stream:
            line = raw_line.decode("utf-8")
            if line.strip() == "<|endoftext|>":
                yield "".join(lines)
                lines.clear()
            else:
                lines.append(line)
        if lines:
            yield "".join(lines)

def fixture_instruction_records(split):
    for story in fixture[120:]:
        key = story_id(story)
        if (story_bucket(key) < 3) != (split == "valid"):
            continue
        first, middle, _ = story.split(". ")
        if int(key[8], 16) % 2:
            yield f"Words: {first.split()[3]}, {middle.split()[-1]}\nStory: {story}"
        else:
            yield f"Random sentence: {middle}.\nStory: {story}"

def parse_instruction(record):
    header, separator, story = record.strip().partition("\nStory:")
    if not separator or not (story := normalize_story(story)):
        return None
    fields = {}
    for line in header.splitlines():
        label, _, value = line.partition(":")
        if value := normalize_story(value):
            fields[label.strip()] = value
    constraints = {}
    if "Words" in fields:
        constraints["words"] = tuple(word.strip().casefold() for word in fields["Words"].split(",") if word.strip())
    if "Random sentence" in fields:
        constraints["sentence"] = fields["Random sentence"]
    if "Summary" in fields:
        constraints["summary"] = fields["Summary"]
    if "Features" in fields:
        constraints["features"] = tuple(FEATURE_NAMES[name.strip()] for name in fields["Features"].split(",")
                                       if name.strip() in FEATURE_NAMES)
    return story, {kind: value for kind, value in constraints.items() if value}

def word_share(words, text):
    found = {word.casefold() for word in WORD.findall(text)}
    return sum(word in found for word in words) / len(words)

def contains_sentence(sentence, text):
    pattern = rf"(?<!\w){re.escape(sentence)}(?!\w)"
    return re.search(pattern, normalize_story(text)) is not None

## Load storytelling requests from TinyStories-Instruct

Each TinyStories-Instruct record combines a story with some of its published instruction fields. A request can name words, quote a sentence that should appear anywhere in the story, describe a plot, or ask for features such as dialogue. The notebook keeps the whole request and story together and drops pairs when an exact word or sentence check fails on the source story. Its CPU fixture uses the same record format with short word and sentence requests.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#prompts-and-system-prompts)

In [ ]:
def instruction_examples(records, limit, excluded_ids, *, training):
    examples = []
    seen = set(excluded_ids)
    for record in records:
        parsed = parse_instruction(record)
        if parsed is None:
            continue
        story, constraints = parsed
        key = story_id(story)
        if not constraints or key in seen or (training and story_bucket(key) >= 18):
            continue
        if "words" in constraints and word_share(constraints["words"], story) < 1:
            continue
        if "sentence" in constraints and not contains_sentence(constraints["sentence"], story):
            continue
        lines = []
        for kind, value in constraints.items():
            clause = ", ".join(value) if isinstance(value, tuple) else value
            lines.append(f"{PROMPT_LABELS[kind]}: {clause}")
        prompt = "\n".join(lines)
        prefix = [BOS, USER, *tokenizer.encode(prompt), ASSISTANT]
        response = [*tokenizer.encode(story), EOS]
        if len(prefix) + len(response) - 1 > cfg.context:
            continue
        examples.append(dict(prompt=prompt, constraints=constraints, prefix=prefix,
                             response=response, reference=story, story_id=key))
        seen.add(key)
        if len(examples) == limit:
            return examples
    raise ValueError(f"Only {len(examples)} of {limit} requested instruction pairs fit the filters.")

instruction_records = fixture_instruction_records if PROFILE == "cpu" else public_instruction_records
base_ids = {story_id(story) for story in pretrain_stories + base_validation}
sft_examples = instruction_examples(
    instruction_records("train"), 360 if PROFILE == "cpu" else cfg.sft_train_size, base_ids, training=True,
)
held_out_prompts = instruction_examples(
    instruction_records("valid"), 40 if PROFILE == "cpu" else 400,
    base_ids | {example["story_id"] for example in sft_examples}, training=False,
)
for split, examples in (("train", sft_examples), ("held-out", held_out_prompts)):
    coverage = Counter(kind for example in examples for kind in example["constraints"])
    expected = {"words", "sentence"} if PROFILE == "cpu" else set(PROMPT_LABELS)
    assert set(coverage) == expected, coverage
    print(f"{split}: {len(examples):,} pairs | {dict(coverage)}")

## Train on the assistant response tokens

Each pair is laid out as a short chat: a start token, a user marker, the request, an assistant marker, the story and an end token. The loss covers only the story and the end token. The first story token is predicted from the assistant marker, and every earlier target is set to the ignore value, so the model learns to answer requests without being trained to write them.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#prompt-loss-masking)

In [ ]:
def instruction_document(example):
    prefix, response = example["prefix"], example["response"]
    inputs, targets = shift_document(prefix + response)
    targets[:len(prefix) - 1] = [IGNORE] * (len(prefix) - 1)
    return inputs, targets

check_example = sft_examples[0]
_, check_targets = instruction_document(check_example)
assert check_targets[len(check_example["prefix"]) - 1] == check_example["response"][0]
assert all(token == IGNORE for token in check_targets[:len(check_example["prefix"]) - 1])
sft_batches, _ = pack_documents(map(instruction_document, sft_examples), cfg.context)

## Fine-tune a copy of the base model

Supervised fine-tuning continues training a copy of the base model on 60,000 instruction pairs for 3,500 updates, at half the pretraining learning rate. The base checkpoint stays unchanged, so the next cells can compare the two models on the same held-out requests.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#supervised-fine-tuning)

In [ ]:
base_model = model
tuned_model = copy.deepcopy(base_model)
sft_run = train(tuned_model, sft_batches, cfg.sft_steps, cfg.lr / 2, "sft-instruct", resume=resume)

## Evaluate held-out storytelling requests

Both checkpoints answer the same held-out requests from the instruction validation file. Response loss measures how well each model predicts the held-out stories given their requests. The generation scores count word and sentence matches in what the model actually writes. Plot summaries and story features need readers to judge meaning. The notebook prints the request and both responses for a small sample, along with a rubric for deciding whether the output followed the requested plot and included each feature.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#intrinsic-and-task-evaluation)

In [ ]:
def lexical_constraints_pass(example, text):
    constraints = example["constraints"]
    words, sentence = constraints.get("words"), constraints.get("sentence")
    return bool(words or sentence) and (
        (not words or word_share(words, text) == 1) and
        (not sentence or contains_sentence(sentence, text))
    )

def compare_models(base_model, tuned_model, examples, limit):
    samples = {kind: [example for example in examples if kind in example["constraints"]]
               for kind in PROMPT_LABELS}
    automatic = ("words", "sentence")
    completions = {}

    def answers(example):
        key = example["story_id"]
        if key not in completions:
            completions[key] = [
                generate(model, example["prefix"], max_new=model.config.context)
                for model in (base_model, tuned_model)
            ]
        return completions[key]

    def show(example):
        base, tuned = answers(example)
        print(f"Request\n{example['prompt']}\nBase\n{base}\nTuned\n{tuned}\n")

    outcomes = {}
    for kind in automatic:
        cases = samples[kind][:limit]
        counts = [0, 0]
        sums = [0.0, 0.0]
        for example in cases:
            constraint = example["constraints"][kind]
            for index, text in enumerate(answers(example)):
                share = (word_share(constraint, text) if kind == "words"
                         else float(contains_sentence(constraint, text)))
                counts[index] += int(share == 1)
                sums[index] += share
        outcomes[kind] = dict(cases=len(cases), passed=counts)
        print(f"{kind}: {counts[0]}/{len(cases)} base | {counts[1]}/{len(cases)} tuned")
        if kind == "words":
            shares = [value / len(cases) for value in sums]
            outcomes[kind]["mean_share"] = shares
            print(f"Mean share of words used: {shares[0]:.1%} base | {shares[1]:.1%} tuned")
        show(cases[0])

    if samples["summary"] or samples["features"]:
        print("Review the plot and features in each answer. Score the plot as a full, partial, or "
              "zero match based on the main events and their outcome. For each requested "
              "feature, mark whether it is present and point to the passage that shows it.")
        review = {example["story_id"]: example for example in samples["summary"][:3]}
        for feature in FEATURE_NAMES.values():
            for example in samples["features"]:
                if feature in example["constraints"]["features"]:
                    review[example["story_id"]] = example
                    break
        for example in review.values():
            show(example)
    return outcomes

validation_batches, _ = pack_documents(map(instruction_document, held_out_prompts), cfg.context)
for name, checkpoint in (("base", base_model), ("tuned", tuned_model)):
    nll, _, targets = score_next_tokens(checkpoint, validation_batches)
    print(f"TinyStories-Instruct {name} response loss: {nll / targets:.3f} nats per token")

tuned_base_metrics = evaluate_language_model(tuned_model, base_validation)
print(f"TinyStories validation perplexity: {base_metrics['perplexity']:.3f} base | "
      f"{tuned_base_metrics['perplexity']:.3f} tuned")
instruction_metrics = compare_models(
    base_model, tuned_model, held_out_prompts, limit=8 if PROFILE == "cpu" else 64,
)

## Compute pass at k from sampled attempts

The formula counts the probability that a sample of k attempts includes at least one success.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#benchmarks-and-pass-at-k)

In [ ]:
def pass_at_k(total, successes, k):
    if not 1 <= k <= total or not 0 <= successes <= total:
        raise ValueError("Require 1 <= k <= total and 0 <= successes <= total.")
    failures = total - successes
    return 1 - math.comb(failures, k) / math.comb(total, k) if failures >= k else 1.0

assert math.isclose(pass_at_k(total=5, successes=2, k=2), 0.7)

## Build auditable preference pairs

For a request that specifies only words, a chosen story uses every word and a rejected story misses one. The pair comes from the instruction training set.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#preference-data-and-reward-models)

In [ ]:
def preference_pair(example):
    chosen = example["reference"]
    word = example["constraints"]["words"][0]
    replacement = "something" if word != "something" else "anything"
    rejected = WORD.sub(lambda match: replacement if match[0].casefold() == word else match[0], chosen)
    assert lexical_constraints_pass(example, chosen) and not lexical_constraints_pass(example, rejected)
    return chosen, rejected

for lab_example in sft_examples:
    if set(lab_example["constraints"]) != {"words"}:
        continue
    chosen_story, rejected_story = preference_pair(lab_example)
    remaining = cfg.context - len(lab_example["prefix"])
    if max(len(tokenizer.encode(chosen_story)), len(tokenizer.encode(rejected_story))) <= remaining:
        break
else:
    raise ValueError("No preference pair fits the context window.")

def softplus(value):
    return torch.logaddexp(value, torch.zeros_like(value))

if RUN_LABS:
    sampled_completions = [
        generate(tuned_model, lab_example["prefix"], temperature=0.7, top_k=25, seed=SEED + index)
        for index in range(6)
    ]
    sampled_completions = [text for text in sampled_completions
                           if 0 < len(tokenizer.encode(text)) <= remaining]
    passing = [text for text in sampled_completions if lexical_constraints_pass(lab_example, text)]
    failing = [text for text in sampled_completions if not lexical_constraints_pass(lab_example, text)]
    if passing and failing:
        chosen_story, rejected_story = passing[0], failing[0]
        print("Using a sampled preference pair.")
    else:
        print("Using a controlled preference pair because the sample lacks both outcomes.")

## Update a policy with DPO

Compare the chosen and rejected log probabilities relative to a frozen copy of the SFT model.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#direct-preference-optimization)

In [ ]:
def completion_logprobs(model, example, response):
    last_available = model.config.context - len(example["prefix"])
    response_ids = tokenizer.encode(response)
    if not response_ids or len(response_ids) > last_available:
        raise ValueError("The completion must fit within the context window.")
    ids = torch.tensor([example["prefix"] + response_ids], device=DEVICE)
    logits, _ = model(ids[:, :-1])
    offset = len(example["prefix"]) - 1
    nll, _ = cross_entropy_loss(logits[:, offset:], ids[:, offset + 1:])
    return -nll[0]

def dpo_loss(policy, reference, example, chosen, rejected, beta=0.1):
    policy_margin = completion_logprobs(policy, example, chosen).sum() - completion_logprobs(policy, example, rejected).sum()
    with torch.no_grad():
        reference_margin = completion_logprobs(reference, example, chosen).sum() - completion_logprobs(reference, example, rejected).sum()
    return softplus(-beta * (policy_margin - reference_margin))

if RUN_LABS:
    dpo_policy = copy.deepcopy(tuned_model)
    dpo_optimizer = AdamW(dpo_policy.named_parameters(), decay=0)
    dpo_optimizer.zero_grad()
    loss = dpo_loss(dpo_policy, tuned_model, lab_example, chosen_story, rejected_story)
    assert math.isclose(loss.item(), math.log(2), rel_tol=1e-5)
    loss.backward()
    dpo_optimizer.step(cfg.lr / 10)
    print("DPO one-pair loss:", round(dpo_loss(dpo_policy, tuned_model, lab_example, chosen_story, rejected_story).item(), 4))

## Compute group advantages and update a policy

Reward passing completions and center rewards within each prompt group. A group whose completions all score alike supplies no advantage.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#verifiable-rewards-and-grpo)

In [ ]:
def group_advantages(rewards):
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    centered = rewards - rewards.mean()
    return centered / (centered.square().mean().sqrt() + 1e-6)

def clipped_policy_loss(current, old, advantage, epsilon):
    ratio = (current - old).exp()
    clipped = ratio.clamp(1 - epsilon, 1 + epsilon)
    return -torch.minimum(ratio * advantage, clipped * advantage).mean()

def grpo_surrogate(policy, old_policy, example, completions, epsilon=0.2):
    rewards = [float(lexical_constraints_pass(example, text)) for text in completions]
    advantages = group_advantages(rewards)
    losses = []
    for text, advantage in zip(completions, advantages):
        current = completion_logprobs(policy, example, text)
        with torch.no_grad():
            old = completion_logprobs(old_policy, example, text)
        losses.append(clipped_policy_loss(current, old, advantage, epsilon))
    return torch.stack(losses).mean(), rewards

assert torch.allclose(group_advantages([1, 0]), torch.tensor([1., -1.], device=DEVICE), atol=1e-5)
if RUN_LABS:
    grpo_policy = copy.deepcopy(tuned_model)
    grpo_optimizer = AdamW(grpo_policy.named_parameters(), decay=0)
    grpo_optimizer.zero_grad()
    sampled = sampled_completions
    if not sampled or len({lexical_constraints_pass(lab_example, text) for text in sampled}) < 2:
        print("Sampled group has no reward variance. Using a controlled pair to demonstrate the objective.")
        sampled = [chosen_story, rejected_story]
    loss, rewards = grpo_surrogate(grpo_policy, tuned_model, lab_example, sampled)
    loss.backward()
    grpo_optimizer.step(cfg.lr / 10)
    print("GRPO group rewards:", rewards)

## Fit a small reward head from pairwise labels

The reward head ranks two completions using pooled token embeddings. This lab trains the scorer used in preference-based RL.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#rlhf)

In [ ]:
class RewardHead(nn.Module):
    def __init__(self, frozen_embeddings):
        super().__init__()
        self.register_buffer("embeddings", frozen_embeddings.detach())
        self.score = Linear(frozen_embeddings.shape[1], 1)

    def forward(self, story):
        ids = torch.tensor(tokenizer.encode(story), device=self.embeddings.device)
        return self.score(self.embeddings[ids].mean(dim=0)).squeeze()

if RUN_LABS:
    reward_model = RewardHead(tuned_model.embedding.weight).to(DEVICE)
    reward_optimizer = AdamW(reward_model.named_parameters(), decay=0)
    for _ in range(20):
        reward_optimizer.zero_grad()
        loss = softplus(reward_model(rejected_story) - reward_model(chosen_story))
        loss.backward()
        reward_optimizer.step(0.01)
    print("Reward head training pair margin:", (reward_model(chosen_story) - reward_model(rejected_story)).item())

## Use learned scores in a clipped policy update

The reward head scores completions, and a fixed policy supplies old log probabilities. This experiment runs one PPO-style policy update with a batch reward baseline.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#rlhf)

In [ ]:
if RUN_LABS and len(set(sampled_completions)) >= 2:
    reward_policy = copy.deepcopy(tuned_model)
    reward_policy_optimizer = AdamW(reward_policy.named_parameters(), decay=0)
    reward_policy_optimizer.zero_grad()
    with torch.no_grad():
        scores = torch.stack([reward_model(text) for text in sampled_completions])
        old_probabilities = [
            completion_logprobs(tuned_model, lab_example, text) for text in sampled_completions
        ]
    losses = []
    for text, old, advantage in zip(sampled_completions, old_probabilities, scores - scores.mean()):
        current = completion_logprobs(reward_policy, lab_example, text)
        losses.append(clipped_policy_loss(current, old, advantage, epsilon=0.2))
    torch.stack(losses).mean().backward()
    reward_policy_optimizer.step(cfg.lr / 10)

## Train on a teacher's soft targets

The student matches the teacher's distribution over possible next tokens. Both models use the same tokenizer.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/post-training-from-base-model-to-assistant/#distillation)

In [ ]:
def distillation_loss(student_logits, teacher_logits, temperature=2):
    student = student_logits.float() / temperature
    teacher_probs = stable_softmax(teacher_logits.detach() / temperature)
    log_probs = student - torch.logsumexp(student, dim=-1, keepdim=True)
    return -(teacher_probs * log_probs).sum(dim=-1).mean() * temperature ** 2

if RUN_LABS:
    from dataclasses import replace

    student = StoryModel(replace(cfg, layers=1)).to(DEVICE)
    student_optimizer = AdamW(student.named_parameters())
    ids = torch.tensor([lab_example["response"][:48]], device=DEVICE)
    with torch.no_grad():
        target, _ = tuned_model(ids)
    predicted, _ = student(ids)
    distillation_loss(predicted, target).backward()
    student_optimizer.step(cfg.lr)

## Track several candidate continuations

Beam search keeps the most likely partial candidates. This simple version scores every prefix from scratch and uses unnormalized total log probability.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#greedy-and-beam-search)

In [ ]:
@torch.inference_mode()
def beam_search(model, prefix, beam_width=3, max_new=12):
    model.eval()
    beams = [((), 0.0, False)]
    for _ in range(min(max_new, model.config.context - len(prefix))):
        candidates = []
        for suffix, score, complete in beams:
            if complete:
                candidates.append((suffix, score, complete))
                continue
            ids = torch.tensor([prefix + list(suffix)], device=DEVICE)
            logits, _ = model(ids)
            log_probs = logits[0, -1].float().clone()
            log_probs[len(tokenizer.tokens):] = -torch.inf
            log_probs[[BOS, USER, ASSISTANT]] = -torch.inf
            log_probs = log_probs - torch.logsumexp(log_probs, dim=-1)
            values, tokens = log_probs.topk(beam_width)
            for candidate_score, token in zip(values.tolist(), tokens.tolist()):
                candidates.append((suffix + (token,), score + candidate_score, token == EOS))
        beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]
        if all(complete for _, _, complete in beams):
            break
    return tokenizer.decode(beams[0][0])

if RUN_LABS:
    print("Beam:", beam_search(tuned_model, lab_example["prefix"]))

## Quantize each weight row to int8

Store one scale per output channel. Dequantize before a matrix multiply and measure the approximation error.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#quantization)

In [ ]:
def quantize_rows(weight):
    scale = (weight.abs().amax(dim=-1, keepdim=True) / 127).clamp_min(1e-8)
    values = (weight / scale).round().clamp(-127, 127).to(torch.int8)
    return values, scale

def dequantize_rows(values, scale):
    return values.float() * scale

with torch.no_grad():
    weight = tuned_model.blocks[0].attention.q.weight
    q_values, q_scale = quantize_rows(weight)
    restored = dequantize_rows(q_values, q_scale)
    assert torch.all((restored - weight).abs() <= q_scale / 2 + 1e-6)

## Train a low-rank update to a frozen matrix

Two small matrices learn the change to a projection. Starting the second matrix at zero preserves the original output at initialization.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#low-rank-adaptation)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=8):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        output_width, input_width = base.weight.shape
        self.a = nn.Parameter(base.weight.new_empty(rank, input_width))
        self.b = nn.Parameter(base.weight.new_zeros(output_width, rank))
        with torch.no_grad():
            self.a.copy_(torch.randn_like(self.a) * 0.02)
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + ((x @ self.a.T) @ self.b.T) * self.scale

if RUN_LABS:
    adapted = copy.deepcopy(base_model).requires_grad_(False)
    attention = adapted.blocks[0].attention
    attention.q = LoRALinear(attention.q)
    original = attention.q.base.weight.detach().clone()
    train(adapted, sft_batches, steps=3, lr=cfg.lr, phase="lora")
    assert torch.equal(attention.q.base.weight, original)
    assert attention.q.b.detach().abs().sum().item() > 0

## Put an adapter on frozen int8 weights

The base projection stores integer values and scales. Only the low-rank update receives gradients.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#quantized-adaptation)

In [ ]:
class QuantizedLoRALinear(nn.Module):
    def __init__(self, adapter):
        super().__init__()
        values, scale = quantize_rows(adapter.base.weight.detach())
        self.register_buffer("values", values)
        self.register_buffer("row_scale", scale)
        self.a, self.b, self.scale = adapter.a, adapter.b, adapter.scale

    def forward(self, x):
        base = x @ dequantize_rows(self.values, self.row_scale).T
        return base + ((x @ self.a.T) @ self.b.T) * self.scale

if RUN_LABS:
    attention.q = QuantizedLoRALinear(attention.q)
    assert sum(p.numel() for p in adapted.parameters() if p.requires_grad) == 8 * cfg.width

## Rank source passages by cosine similarity

A transparent bag-of-words vector makes retrieval easy to inspect. Use the ranked passages as context for a story prompt.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#embedding-models-and-vector-search)

In [ ]:
def cosine_rank(query, passages, limit=2):
    vectors = [Counter(WORD.findall(text.casefold())) for text in [query, *passages]]
    query_vector, *passage_vectors = vectors
    def cosine(a, b):
        numerator = sum(a[word] * b[word] for word in a)
        denominator = math.sqrt(sum(n*n for n in a.values()) * sum(n*n for n in b.values()))
        return numerator / denominator if denominator else 0.0
    ranked = sorted(zip(passages, passage_vectors), key=lambda item: cosine(query_vector, item[1]), reverse=True)
    return [passage for passage, _ in ranked[:limit]]

passages = ["The fox hides under the bridge.", "An owl sleeps by the lake.", "The bridge is green."]
assert cosine_rank("fox bridge", passages)[0] == passages[0]

## Run a bounded story lookup tool

The host validates the tool and its argument, then adds a result to the context for the story. The notebook supplies a fixed call to demonstrate the protocol.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#tool-calling)

In [ ]:
def dispatch_story_tool(call, library):
    if call.get("tool") != "find_passage" or not isinstance(call.get("query"), str):
        raise ValueError("Expected a find_passage call with a text query.")
    return cosine_rank(call["query"][:120], library, limit=1)

def build_grounded_prompt(instruction, tool_call, library):
    passages = dispatch_story_tool(tool_call, library)
    context = "\n".join(f"[{i}] {passage}" for i, passage in enumerate(passages, 1))
    return f"Source passages:\n{context}\n\nRequest: {instruction}"

grounded = build_grounded_prompt("Write about the fox.", {"tool": "find_passage", "query": "fox bridge"}, passages)
assert passages[0] in grounded and passages[1] not in grounded

## Check cited source numbers

A citation must point at a passage the application actually provided. The check leaves claim support to a separate evaluation.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#grounding-and-citation-checks)

In [ ]:
def citations_resolve(response, source_count):
    citations = [int(value) for value in re.findall(r"\[(\d+)\]", response)]
    return bool(citations) and all(1 <= source <= source_count for source in citations)

assert citations_resolve("The bridge is green [1].", 1)
assert not citations_resolve("The bridge is green [2].", 1)

## Compare predicted confidence with accuracy

Expected calibration error summarizes the gap between average confidence and observed accuracy across bins.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#calibration-and-uncertainty)

In [ ]:
def calibration_error(confidences, correct, bins=5):
    if len(confidences) != len(correct) or not confidences or bins < 1:
        raise ValueError("Confidence and correctness must be paired in at least one bin.")
    grouped = [[] for _ in range(bins)]
    for confidence, hit in zip(confidences, correct):
        if not 0 <= confidence <= 1:
            raise ValueError("Confidence must be a probability.")
        grouped[min(int(confidence * bins), bins-1)].append((confidence, hit))
    return sum(abs(sum(conf for conf, _ in group) - sum(hit for _, hit in group))
               for group in grouped) / len(confidences)

assert math.isclose(calibration_error([0.6, 0.6], [True, False]), 0.1)

## Check exact story overlap between stages

Content hashes catch identical stories even if the original spacing or capitalization differs.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#benchmark-contamination)

In [ ]:
def story_ids(stories):
    return {story_id(text) for text in stories}

assert story_id("A fox. ") == story_id("a FOX.")
pretrain_ids = story_ids(pretrain_stories)
base_validation_ids = story_ids(base_validation)
instruction_train_ids = {example["story_id"] for example in sft_examples}
instruction_eval_ids = {example["story_id"] for example in held_out_prompts}

assert pretrain_ids.isdisjoint(base_validation_ids)
assert base_ids.isdisjoint(instruction_train_ids | instruction_eval_ids)
assert instruction_train_ids.isdisjoint(instruction_eval_ids)

## Ablate one attention output and measure the change

Zeroing the first block's attention output gives one controlled comparison on a small set of validation stories.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#interpretability)

In [ ]:
if RUN_LABS:
    short_eval = base_validation[:8]
    baseline = evaluate_language_model(base_model, short_eval)
    layer = base_model.blocks[0].attention.out
    handle = layer.register_forward_hook(lambda _layer, _inputs, output: torch.zeros_like(output))
    try:
        ablated = evaluate_language_model(base_model, short_eval)
    finally:
        handle.remove()
    print("First attention output ablation, loss before/after:", baseline["loss"], ablated["loss"])

## Score a story by adding next-token log probabilities

The model assigns a conditional probability to each observed token. Adding the log probabilities scores the whole sequence.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/language-modeling-and-next-token-prediction/#the-chain-rule-of-probability)

In [ ]:
@torch.inference_mode()
def sequence_log_probability(model, text):
    inputs, targets = shift_document(tokenizer.document(text))
    if len(inputs) > model.config.context:
        raise ValueError("This example scores one context window at a time.")
    x = torch.tensor([inputs], device=DEVICE)
    y = torch.tensor([targets], device=DEVICE)
    logits, _ = model(x)
    losses, _ = cross_entropy_loss(logits, y)
    return -losses.sum().item()

assert sequence_log_probability(base_model, "A fox crossed the bridge.") < 0

## Read one embedding row

Indexing by a token ID gives the same vector as multiplying the table by a one-hot row.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/tokenization-and-embeddings/#embedding-lookup)

In [ ]:
with torch.no_grad():
    table = model.embedding.weight
    example_id = tokenizer.encode("fox")[0]
    one_hot = table.new_zeros(table.shape[0])
    one_hot[example_id] = 1
    torch.testing.assert_close(model.embedding(torch.tensor(example_id, device=DEVICE)), one_hot @ table, atol=1e-6, rtol=1e-5)

## Compare token counts on the same validation text

Fertility is the ratio of token count to word count. Byte-level fallback increases the ratio for text with unfamiliar spellings.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/tokenization-and-embeddings/#fertility)

In [ ]:
def tokens_per_word(stories, encode):
    total_words = sum(len(story.split()) for story in stories)
    return sum(len(encode(story)) for story in stories) / total_words

contrast = base_validation[:12]
bpe_fertility = tokens_per_word(contrast, tokenizer.encode)
byte_fertility = tokens_per_word(contrast, lambda text: text.encode("utf-8"))
assert bpe_fertility <= byte_fertility

## Check that future and neighboring documents stay invisible

Changing a hidden token must leave the earlier document's logits unchanged. The same check catches a dropped packing mask.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#token-mixing)

In [ ]:
with torch.inference_mode():
    probe = torch.tensor([[BOS, 65, EOS, BOS, 66, EOS]], device=DEVICE)
    segments = torch.tensor([[0, 0, 0, 1, 1, 1]], device=DEVICE)
    original, _ = base_model(probe, segments)
    probe[0, 1] = 67
    changed, _ = base_model(probe, segments)
    torch.testing.assert_close(original[:, 0], changed[:, 0])
    torch.testing.assert_close(original[:, 3:], changed[:, 3:])

## Measure how dot product spread changes with width

For independent unit-scale coordinates, the unscaled score spread grows with head width.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#square-root-scaling)

In [ ]:
generator = torch.Generator().manual_seed(SEED)
for head_width in (4, 64):
    queries = torch.randn(2048, head_width, generator=generator)
    keys = torch.randn(2048, head_width, generator=generator)
    score = (queries * keys).sum(dim=-1)
    assert 0.9 < (score / math.sqrt(head_width)).std().item() < 1.1

## Build an additive position table

The sine and cosine columns encode each absolute position at several frequencies.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#absolute-and-sinusoidal-positions)

In [ ]:
def sinusoidal_positions(length, width, base=10_000.0):
    exponent = torch.arange(0, width, 2, dtype=torch.float32) / width
    phase = torch.arange(length)[:, None] * base ** -exponent
    return torch.stack((phase.sin(), phase.cos()), dim=-1).flatten(1)[:, :width]

assert sinusoidal_positions(3, 4)[0].tolist() == [0., 1., 0., 1.]

## Build a linear distance bias for attention

Different heads can use different slopes. A larger slope reduces the score of distant positions faster.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/attention-and-positional-information/#alibi-and-length-extrapolation)

In [ ]:
def alibi_bias(slopes, length):
    positions = torch.arange(length, device=slopes.device)
    distance = positions[:, None] - positions[None, :]
    return -slopes[:, None, None] * distance.clamp_min(0)

assert alibi_bias(torch.tensor([0.5]), 3)[0, 2].tolist() == [-1., -0.5, 0.]

## Check the residual path with disabled branches

Setting the two output projections to zero leaves the block as the identity map.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#the-residual-stream)

In [ ]:
identity_block = Block(CPU).eval()
with torch.no_grad():
    identity_block.attention.out.weight.zero_()
    identity_block.feed_forward.down.weight.zero_()
    stream = torch.randn(1, 3, CPU.width)
    output, _ = identity_block(stream, positions[None], causal_mask(positions, positions)[None])
    torch.testing.assert_close(output, stream)

## Compare compute at two context lengths

Parameter count stays fixed when context grows. The dense attention terms cost more per token as sequences get longer.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#flops-per-token)

In [ ]:
from dataclasses import replace

short = model_accounting(replace(STORY, context=128))
long = model_accounting(STORY)
assert short["parameters"] == long["parameters"] == 11_015_040
assert long["forward_flops_per_token"] > short["forward_flops_per_token"]
assert long["cache_bytes"] == 4 * short["cache_bytes"] == 1_572_864
print("Story profile estimates:", long)

## Route tokens to their top experts

This small reference layer uses the same SwiGLU inside each expert. Only experts selected by the router see each token.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#mixture-of-experts)

In [ ]:
class MixtureOfExperts(nn.Module):
    def __init__(self, width, hidden, n_experts=4, k=2):
        super().__init__()
        self.router = Linear(width, n_experts)
        self.experts = nn.ModuleList([SwiGLU(width, hidden) for _ in range(n_experts)])
        self.k = k

    def forward(self, inputs):
        rows = inputs.reshape(-1, inputs.shape[-1])
        scores, decisions = self.router(rows).topk(self.k, dim=-1)
        weights = stable_softmax(scores)
        output = torch.zeros_like(rows)
        for expert_index, expert in enumerate(self.experts):
            tokens, slots = (decisions == expert_index).nonzero(as_tuple=True)
            values = expert(rows[tokens]) * weights[tokens, slots, None]
            output.index_add_(0, tokens, values.to(output.dtype))
        return output.reshape_as(inputs), decisions

if RUN_LABS:
    routed, picked = MixtureOfExperts(16, 24)(torch.ones(1, 3, 16))
    assert routed.shape == (1, 3, 16) and picked.shape == (3, 2)

## Sample two sources at a chosen ratio

Source weights control how often the iterator draws from each pool. This experiment uses fixture labels to make the realized mix observable.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/pretraining-at-scale/#data-mixtures)

In [ ]:
def sample_mixture(sources, weights, count, seed=SEED):
    rng = random.Random(seed)
    for source in rng.choices(range(len(sources)), weights=weights, k=count):
        yield source, rng.choice(sources[source])

counts = Counter(source for source, _ in sample_mixture(
    [pretrain_stories[:10], pretrain_stories[10:20]], [0.8, 0.2], 100
))
assert counts[0] > counts[1]

## Turn cache shape into a memory budget

One active request stores keys and values for each layer and position. The two KV heads keep the cache smaller than storing a copy for every query head.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#memory-bandwidth-bound-decoding)

In [ ]:
story_bytes = model_accounting(STORY, dtype_bytes=2)
mha_bytes = model_accounting(replace(STORY, kv_heads=STORY.heads), dtype_bytes=2)
assert mha_bytes["cache_bytes"] == 3 * story_bytes["cache_bytes"]
print(f"{story_bytes['cache_bytes'] / 2**20:.1f} MiB per full-context bf16 cache")

## Time prefill and one cached decode separately

Prefill processes all prompt positions at once. Decode adds a token to an existing cache.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#prefill-and-decode-phases)

In [ ]:
@torch.inference_mode()
def timing_probe(model, prompt, repeats=12):
    ids = torch.tensor([[BOS, *tokenizer.encode(prompt)]], device=DEVICE)
    next_id = torch.tensor([[65]], device=DEVICE)
    _, cache = model(ids, use_cache=True)
    def synchronize():
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elif DEVICE.type == "mps":
            torch.mps.synchronize()
    def timed(fn):
        synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            fn()
        synchronize()
        return (time.perf_counter() - start) * 1000 / repeats
    return dict(prefill_ms=timed(lambda: model(ids, use_cache=True)),
                decode_ms=timed(lambda: model(next_id, cache=cache, use_cache=True)))

if RUN_LABS:
    print(timing_probe(base_model, "One day a fox crossed a bridge."))

## Accept a draft token or sample the corrected distribution

The target accepts a draft token with probability min(1, p/q). On rejection, the positive difference between target and draft distributions supplies the next token.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#speculative-decoding)

In [ ]:
def speculative_one_token(target_probs, draft_probs, draft_token, draw, replacement_draw):
    if draw <= min(1.0, (target_probs[draft_token] / draft_probs[draft_token]).item()):
        return draft_token, True
    corrected = (target_probs - draft_probs).clamp_min(0)
    cumulative = (corrected / corrected.sum()).cumsum(0)
    return torch.searchsorted(cumulative, cumulative.new_tensor(replacement_draw), right=True).item(), False

assert speculative_one_token(torch.tensor([0.2, 0.8]), torch.tensor([0.5, 0.5]), 0, 0.9, 0.7) == (1, False)

## Map logical KV pages to physical slots

The page table lets two requests grow independently while sharing the same pool of physical storage.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#continuous-batching-and-paged-attention)

In [ ]:
def page_slots(logical_block_table, token_positions, page_size):
    return [logical_block_table[position // page_size] * page_size + position % page_size
            for position in token_positions]

assert page_slots([3, 1], [0, 1, 2, 3, 4], page_size=4) == [12, 13, 14, 15, 4]

## Build a context budget for retrieved passages

Retrieval is a host operation. The host adds only the passages that leave enough context for generation.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#retrieval-augmented-generation)

In [ ]:
def retrieved_prefix(question, library, limit=2, budget=128):
    prompt = f"Request: {question}"
    prefix = [BOS, USER, *tokenizer.encode(prompt), ASSISTANT]
    if len(prefix) > budget:
        raise ValueError("The question alone exceeds the prompt budget.")
    context = ""
    for i, passage in enumerate(cosine_rank(question, library, limit), 1):
        candidate = context + f"Source [{i}] {passage}\n"
        token_ids = [BOS, USER, *tokenizer.encode(candidate + prompt), ASSISTANT]
        if len(token_ids) <= budget:
            prefix, context = token_ids, candidate
    return prefix

assert len(retrieved_prefix("fox bridge", passages, budget=64)) <= 64

## Flag verbatim repetition of a synthetic canary

A unique, fabricated string helps check for exact copying without using anyone's personal data.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/evaluation-reliability-and-safety/#memorization-privacy-and-bias)

In [ ]:
def contains_canary(generation, canary):
    return normalize_story(canary).casefold() in normalize_story(generation).casefold()

assert contains_canary("The code was coral   racket nine.", "CORAL RACKET NINE")
assert not contains_canary("The code was nine.", "CORAL RACKET NINE")

## Compare centering with RMS normalization

LayerNorm subtracts the mean before rescaling. RMSNorm divides by the root mean square without centering.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/the-transformer-block-and-model-anatomy/#layer-normalization-and-rmsnorm)

In [ ]:
def layer_norm_reference(x, eps=1e-5):
    centered = x - x.mean(dim=-1, keepdim=True)
    variance = centered.square().mean(dim=-1, keepdim=True)
    return centered * torch.rsqrt(variance + eps)

example = torch.tensor([[2., 4.]])
assert abs(layer_norm_reference(example).mean().item()) < 1e-6
assert RMSNorm(2)(example).mean().item() > 0

## Mask illegal next tokens before sampling

A decoder can use the current parser state to remove illegal next tokens. This probe limits the first answer token to two allowed choices.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/inference-and-decoding/#repetition-control-stopping-and-grammar-masks)

In [ ]:
def allow_only(logits, allowed):
    if not allowed:
        raise ValueError("The grammar must allow at least one next token.")
    permitted = torch.zeros_like(logits, dtype=torch.bool)
    permitted[allowed] = True
    return logits.masked_fill(~permitted, -torch.inf)

assert allow_only(torch.tensor([9., 2., 3.]), [1, 2]).argmax().item() == 2

## Feed a tool result back into the next turn

The host owns the step budget. This fixture follows one structured tool call and stops at a final answer.

[Read the tutorial section](https://cook-ml.com/tutorials/llm-fundamentals/using-llms-in-applications/#the-agent-loop)

In [ ]:
def bounded_agent(next_message, library, max_turns=3):
    history = []
    for _ in range(max_turns):
        message = next_message(history)
        if "answer" in message:
            return message["answer"]
        history.append(dispatch_story_tool(message, library))
    raise RuntimeError("The tool loop reached its turn limit.")

def scripted_turn(history):
    return {"answer": history[-1][0]} if history else {"tool": "find_passage", "query": "fox bridge"}

assert bounded_agent(scripted_turn, passages) == passages[0]